# Schelling segregation on the skabm machinery

The point of this notebook is what it does *not* contain: no space module, no
scheduler, no agent classes, no visitation loop — and no agent population
either. The caller declares only the **grid** (a `Cell` population); the people
are derived in SPARQL by `schelling.SETTLE`, which occupies each cell with
probability `$density` and assigns a random group, exactly as Poledna's
`firm_ownership` hands out firm ownership for free.

Everything else — the Moore neighbourhood, happiness, the move — is a
`string.Template` rule in [`skabm/behaviour/schelling.py`](../skabm/behaviour/schelling.py), run by
the same `RDFSimulator.fit_iter` that drives the macro ABM.

Compare with the AMBER reference (`examples/segregation_model.py` there):
`Model.grid`, `Model.empty_spots`, `get_neighbors`, `Person.find_new_home`, the
placement loop in `setup`, and the sequential sweep all collapse into
declarative graph patterns.

The continuous-geometry variant — the same rules over GeoSPARQL points instead
of a lattice, swapping only `def:neighbor` — is covered by
`tests/test_schelling.py::test_geo_variant_plugs_in`.

In [ ]:
from time import time

import matplotlib.pyplot as plt
import polars as pl
from maplib import Model

from skabm.behaviour.schelling import (
    PARAMETERS,
    SCHELLING_INIT_RULES,
    SCHELLING_UPDATE_RULES,
    state_extract,
    want_similar,
)
from skabm.simulation import RDFSimulator
from skabm.ottr import cell_template  # + DataFrame.with_iri

SIZE = 50  # grid side; 2500 cells, ~2000 settled at density 0.8

## The grid is a population

Purely spatial: coordinates and an id, nothing random. `x`/`y` are `Float64`
because SPARQL basic graph patterns join on RDF *terms*, so the `xsd:double`
produced by `?x + ?dx` in `GRID_NEIGHBORHOOD` has to meet stored `xsd:double`s —
an `xsd:integer` would match nothing, silently.

In [ ]:
cells = (
    pl.DataFrame({"x": range(SIZE)})
    .join(pl.DataFrame({"y": range(SIZE)}), how="cross")
    .with_columns(
        pl.format("cell_{}_{}", pl.col("x"), pl.col("y")).alias("id"),
        pl.col("x").cast(pl.Float64),
        pl.col("y").cast(pl.Float64),
    )
)
cells.head()

## The simulator

`density` and `n_groups` default to the module's `PARAMETERS`, overridable
through `params=`; `random_seed` pins the `pr:uniform` draws. Only the rule tuples, the
params and the state extract differ from the Poledna configuration — the
simulator itself is unchanged.

In [ ]:
sim = RDFSimulator(
    init_rules=SCHELLING_INIT_RULES,
    update_rules=SCHELLING_UPDATE_RULES,
    n_periods=40,
    state_extract=state_extract,
    random_seed=0,
)

## Run

AMBER's `get_segregation()` is yielded as an observable; the unhappy count is a
property of the *distribution*, so it comes off the per-agent frame that
`sim.extract()` returns inside the loop body.

`RDFSimulator` has no convergence stop — it runs exactly `n_periods`. The
generator is where that belongs anyway: the caller owns the criterion.

In [ ]:
t0 = time()
history = []

world = Model()
world.map(cell_template, cells.with_iri())
for row in sim.fit_iter(world):
    t = row["t"]
    unhappy = int(
        (sim.extract()["share_similar"] < PARAMETERS[want_similar]).sum()
    )
    segregation = row["sig__AVG__Person__share_similar"]
    history.append({"t": t, "segregation": segregation, "unhappy": unhappy})
    print(t, round(segregation, 3), unhappy)
    if unhappy == 0:
        print(f"converged at t={t}")
        break

elapsed = time() - t0
print(f"{t} ticks in {elapsed:.2f}s ({elapsed / t:.3f} s/tick)")

history = pl.DataFrame(history)
history

## The segregation curve

Segregation rises monotonically while the unhappy count falls to zero: every
move is made by a dissatisfied agent, and a move that satisfies it raises the
mean share of same-group neighbours. The classic result, with the threshold at
3 of 8 Moore neighbours.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history["t"], history["segregation"], marker="o", color="tab:blue")
ax.set_xlabel("tick")
ax.set_ylabel("mean share of similar neighbours", color="tab:blue")
ax.tick_params(axis="y", labelcolor="tab:blue")

unhappy_ax = ax.twinx()
unhappy_ax.plot(history["t"], history["unhappy"], marker="s", color="tab:red")
unhappy_ax.set_ylabel("unhappy agents", color="tab:red")
unhappy_ax.tick_params(axis="y", labelcolor="tab:red")

ax.set_title(f"Schelling on a {SIZE}x{SIZE} lattice")
fig.tight_layout()